In [5]:
print("Ready")

Ready


# Advanced RAG Series — Video 04 (2026 Edition)
# MergerRetriever & LongContextReorder: Solving the Lost-in-the-Middle Phenomenon

**Channel:** IT AI Enthusiast | @itaienthusiast  
**GitHub:** [mayankchugh-learning](https://github.com/mayankchugh-learning)  
**Runtime:** Google Colab T4 GPU (recommended)  

> **Paper:** Liu et al., *Lost in the Middle: How Language Models Use Long Contexts*,  
> Stanford / UC Berkeley / Samaya AI, TACL 2023. arXiv:2307.03172  

> **2026 Updates:** BAAI/bge-m3 embeddings · gpt-4o-mini LLM · langchain_classic imports · LCEL chain  
> **Corpus note:** This notebook loads *Alice's Adventures in Wonderland* (Lewis Carroll, 1865 —
> public domain worldwide) directly from a real PDF via `PyPDFLoader`, split into two thematic
> collections by chapter. (An earlier draft used the 3_Idiots.pdf screenplay, which is
> copyrighted by Vinod Chopra Films / Om Books International and is **not** used here.)


## 1 — The Lost-in-the-Middle Problem (from the paper)

**Liu et al. 2023** (arXiv:2307.03172) measured multi-document QA accuracy as a function  
of where the relevant document was positioned among 10/20/30 retrieved documents:

| Position | GPT-3.5-Turbo Accuracy |
|----------|------------------------|
| Position 1 (start) | ~75.8% |
| Positions 5–14 (middle) | 53–57% |
| Position 19–20 (end) | ~63.2% |
| Closed-book (no docs) | 56.1% |

> **U-shaped curve:** Best at start and end. Worst in the middle — sometimes worse than no context at all.

**Mitigations in this notebook:**
- `MergerRetriever` — domain-separated retrieval, merged pool
- `EmbeddingsRedundantFilter` — remove near-duplicates
- `LongContextReorder` — push best docs to positions 1 & N

## 2 — Install Dependencies (2026 Edition)

In [6]:
# ── Install dependencies (2026 Edition — run once) ───────────────────────
%pip install -q \
    langchain langchain-classic langchain-community \
    langchain-chroma langchain-huggingface langchain-openai \
    langchain-text-splitters chromadb sentence-transformers \
    transformers pypdf

print('Dependencies installed — 2026 edition')


Dependencies installed — 2026 edition


In [7]:
# ── Verify key package versions ───────────────────────────────────────────
import importlib

for pkg in [
    "langchain",
    "langchain_classic",
    "langchain_community",
    "langchain_chroma",
    "langchain_huggingface",
    "chromadb",
    "sentence_transformers",
]:
    try:
        mod = importlib.import_module(pkg)
        print(f"  {pkg:<28} {getattr(mod, '__version__', 'ok')}")
    except ImportError as exc:
        print(f"  {pkg:<28} MISSING ({exc})")


  langchain                    1.3.17
  langchain_classic            1.0.8
  langchain_community          0.4.2
  langchain_chroma             ok
  langchain_huggingface        ok
  chromadb                     1.5.9
  sentence_transformers        5.7.0


## 3 — Load Documents

Upload copy of alice-in-wonderland.pdf

## 2 — Load documents

In [8]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Point this at your uploaded copy of alice-in-wonderland.pdf
ALICE_PDF = os.getenv("ALICE_PDF", "/content/alice-in-wonderland.pdf")

if not Path(ALICE_PDF).exists():
    raise FileNotFoundError(
        f"Could not find {ALICE_PDF}. Upload alice-in-wonderland.pdf to Colab "
        "(or set the ALICE_PDF environment variable to its path) before continuing."
    )

print(f"Loading PDF: {ALICE_PDF}")
all_pages = PyPDFLoader(ALICE_PDF).load()
print(f"Total pages loaded: {len(all_pages)}")

# Auto-detect the split point: "Chapter VII" marks the start of the trial/croquet half.
split_index = next(
    (i for i, p in enumerate(all_pages) if "Chapter VII" in p.page_content),
    len(all_pages) // 2,  # fallback: split the book in half
)

early_docs = all_pages[:split_index]   # Chapters I-VI: Rabbit-Hole -> Pig and Pepper
court_docs = all_pages[split_index:]   # Chapters VII-XII: Mad Tea-Party -> Alice's Evidence

for doc in early_docs:
    doc.metadata["source"] = "wonderland_early"
for doc in court_docs:
    doc.metadata["source"] = "wonderland_court"

print(f"Early-chapters pages (I-VI)   : {len(early_docs)}")
print(f"Court-chapters pages (VII-XII): {len(court_docs)}")


Loading PDF: /content/alice-in-wonderland.pdf
Total pages loaded: 66
Early-chapters pages (I-VI)   : 32
Court-chapters pages (VII-XII): 34


## 3 — Chunk the documents

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

early_chunks = text_splitter.split_documents(early_docs)
court_chunks = text_splitter.split_documents(court_docs)

print(f"Wonderland (early) chunks : {len(early_chunks)}")
print(f"Wonderland (court) chunks : {len(court_chunks)}")
print("\nExample chunk:\n", early_chunks[0].page_content[:200], "...")


Wonderland (early) chunks : 195
Wonderland (court) chunks : 190

Example chunk:
 Alice’s Adventures in Wonderland 
 
Lewis Carroll 
 
 
 
With illustrations by John Tenniel 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
This .pdf file was made available through Lenny’s Alice in Wo ...


## 4 — Embedding Model (2026: BAAI/bge-m3)

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# 2026 update: BAAI/bge-m3 replaces bge-large-en
# Multilingual, 1024-dim, top MTEB performer, same API
EMBEDDING_MODEL = 'BAAI/bge-m3'

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print(f'Embedding model loaded: {EMBEDDING_MODEL}')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-m3


## 5 — Index into Chroma (two collections)

In [11]:
import shutil
from langchain_chroma import Chroma

DB_DIR = Path("./chroma_merger_reorder_db")
if DB_DIR.exists():
    shutil.rmtree(DB_DIR)
DB_DIR.mkdir(parents=True, exist_ok=True)

early_vectorstore = Chroma.from_documents(
    documents=early_chunks,
    embedding=embeddings,
    collection_name="wonderland_early",
    persist_directory=str(DB_DIR / "wonderland_early"),
    collection_metadata={"hnsw:space": "cosine"},
)

court_vectorstore = Chroma.from_documents(
    documents=court_chunks,
    embedding=embeddings,
    collection_name="wonderland_court",
    persist_directory=str(DB_DIR / "wonderland_court"),
    collection_metadata={"hnsw:space": "cosine"},
)

print("Vector stores ready:", DB_DIR.resolve())


Vector stores ready: /content/chroma_merger_reorder_db


## 6 — Per-domain retrievers (MMR)

In [12]:
retriever_early = early_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 12},
)

retriever_court = court_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 12},
)


## 6 — Per-domain retrievers (MMR)

In [13]:
retriever_early = early_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 12},
)

retriever_court = court_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 12},
)


## 7 — MergerRetriever (Lord of the Retrievers / LOTR)

In [14]:
from langchain_classic.retrievers import MergerRetriever

lotr = MergerRetriever(retrievers=[retriever_early, retriever_court])

query = "Who is the Queen of Hearts?"
merged_docs = lotr.invoke(query)

print(f"Query: {query}")
print(f"Merged chunks returned: {len(merged_docs)}\n")
for i, doc in enumerate(merged_docs, 1):
    preview = doc.page_content.replace("\n", " ")[:120]
    print(f"{i}. [{doc.metadata.get('source', 'unknown')}] {preview}...")


Query: Who is the Queen of Hearts?
Merged chunks returned: 10

1. [wonderland_early] invitation from the Queen to play croquet.' The Frog-Footman  repeated, in the same solemn tone, only changing the order...
2. [wonderland_court] 40 severely `Who is this?' She said it to the Knave of Hearts, who only bowed and smiled in reply.     `Idiot!' said the...
3. [wonderland_early] seven is--oh dear! I shall never get to twenty at that rate! However, the Multiplication Table doesn't  signify: let's t...
4. [wonderland_court] was said, and went by without noticing her. Then followed the Knave of Hearts, carrying the King's  crown on a crimson v...
5. [wonderland_early] dead silence.     Alice noticed with some surprise that the pebbles were all turning into little cakes as they lay on th...
6. [wonderland_court] 56 Chapter XI: Who Stole the Tarts?    The King and Queen of Hearts were seated on their throne when they arrived, with ...
7. [wonderland_early] 16 `I wish I hadn't mentioned Dinah!' s

In [15]:
query = "Who is the Caterpillar?"
merged_docs = lotr.invoke(query)

print(f"Query: {query}")
print(f"Merged chunks returned: {len(merged_docs)}\n")
for i, doc in enumerate(merged_docs, 1):
    preview = doc.page_content.replace("\n", " ")[:120]
    print(f"{i}. [{doc.metadata.get('source', 'unknown')}] {preview}...")


Query: Who is the Caterpillar?
Merged chunks returned: 10

1. [wonderland_early] 21 Chapter V: Advice from a Caterpillar    The Caterpillar and Alice looked at each other for some time in silence: at l...
2. [wonderland_court] WILLIAM,' to the Caterpillar, and the words all coming different, and then the Mock Turtle drew a  long breath, and said...
3. [wonderland_early] minutes.     The Caterpillar was the first to speak.     `What size do you want to be?' it asked.     `Oh, I'm not parti...
4. [wonderland_court] little sisters--they were learning to draw, you know--'     `What did they draw?' said Alice, quite forgetting her promi...
5. [wonderland_early] you?'     `I'm afraid I am, sir,' said Alice; `I can't remember things as I used--and I don't keep the same size for  te...
6. [wonderland_court] and then, `we went to school in the sea. The master was an old Turtle--we used to call him Tortoise--'     `Why did you ...
7. [wonderland_early] begin with; and being so many different siz

> **Observation:** Raw merged results are noisy and cross-domain. The next section builds the filter + reorder pipeline to fix this and mitigate lost-in-the-middle.

## 8 — Compression pipeline: dedupe + reorder

In [16]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import (
    EmbeddingsRedundantFilter,
    LongContextReorder,
)

redundancy_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
reordering = LongContextReorder()

compressor_pipeline = DocumentCompressorPipeline(
    transformers=[redundancy_filter, reordering]
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor_pipeline,
    base_retriever=lotr,
)

print("Compression retriever ready.")


Compression retriever ready.


In [17]:
def show_retrieval(label: str, query: str, retriever) -> None:
    docs = retriever.invoke(query)
    print(f"\n{'=' * 72}")
    print(f"{label}")
    print(f"Query: {query}")
    print(f"Documents returned: {len(docs)}")
    print('=' * 72)
    for i, doc in enumerate(docs, 1):
        preview = doc.page_content.replace("\n", " ")[:140]
        print(f"{i}. {preview}...")

show_retrieval("Raw MergerRetriever (no compression)", "Who is the Queen of Hearts?", lotr)
show_retrieval("After redundant filter + LongContextReorder", "Who is the Queen of Hearts?", compression_retriever)



Raw MergerRetriever (no compression)
Query: Who is the Queen of Hearts?
Documents returned: 10
1. invitation from the Queen to play croquet.' The Frog-Footman  repeated, in the same solemn tone, only changing the order of  the words a lit...
2. 40 severely `Who is this?' She said it to the Knave of Hearts, who only bowed and smiled in reply.     `Idiot!' said the Queen, tossing her ...
3. seven is--oh dear! I shall never get to twenty at that rate! However, the Multiplication Table doesn't  signify: let's try Geography. London...
4. was said, and went by without noticing her. Then followed the Knave of Hearts, carrying the King's  crown on a crimson velvet cushion; and, ...
5. dead silence.     Alice noticed with some surprise that the pebbles were all turning into little cakes as they lay on the  floor, and a brig...
6. 56 Chapter XI: Who Stole the Tarts?    The King and Queen of Hearts were seated on their throne when they arrived, with a great crowd  assem...
7. 16 `I wish I hadn't

## 9 — LLM Setup (2026: gpt-4o-mini or Qwen2.5-7B local)

In [19]:
import os
from google.colab import userdata

# 2026 update: gpt-4o-mini replaces Zephyr-7B/llama.cpp
# No API key? Falls back to Qwen2.5-7B-Instruct locally
OPENAI_API_KEY = '' # Initialize with an empty string

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except userdata.SecretNotFoundError:
    print("OPENAI_API_KEY not found in Colab secrets. Falling back to local model.")
    OPENAI_API_KEY = '' # Ensure it's empty if not found

llm = None
if OPENAI_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2, api_key=OPENAI_API_KEY)
    print('LLM: ChatOpenAI gpt-4o-mini')
else:
    from langchain_huggingface import HuggingFacePipeline
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    print('No OPENAI_API_KEY — loading Qwen2.5-7B-Instruct locally...')
    MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto')
    pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=512)
    llm = HuggingFacePipeline(pipeline=pipe)
    print(f'LLM: {MODEL_ID} (local)')

LLM: ChatOpenAI gpt-4o-mini


## 10 — RAG Chain (LCEL — 2026 Pattern)

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    return "\n\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(docs, 1))


prompt = PromptTemplate.from_template(
    """Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:"""
)

rag_chain = (
    {
        "context": compression_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain ready.")


LCEL RAG chain ready.


## 11 — Run test queries

In [21]:
def ask(question: str):
    print(f"\nQuestion: {question}")
    answer = rag_chain.invoke(question)
    print(f"Answer:\n{answer}\n")
    sources = compression_retriever.invoke(question)
    print(f"Source chunks used ({len(sources)}):")
    for i, doc in enumerate(sources, 1):
        print(f"  {i}. {doc.metadata.get('source', 'n/a')} — {doc.page_content[:100]}...")

ask("Who is the Queen of Hearts?")



Question: Who is the Queen of Hearts?
Answer:
I do not know.

Source chunks used (10):
  1. wonderland_court — 40
severely `Who is this?' She said it to the Knave of Hearts, who only bowed and smiled in reply.  ...
  2. wonderland_court — was said, and went by without noticing her. Then followed the Knave of Hearts, carrying the King's 
...
  3. wonderland_court — 56
Chapter XI: Who Stole the Tarts? 
 
The King and Queen of Hearts were seated on their throne when...
  4. wonderland_court — this was of very little use, as it left no mark on the slate.  
 
`Herald, read the accusation!' sai...
  5. wonderland_court — in particular; at least, if there are, nobody 
attends to them--and you've no idea how 
confusing it...
  6. wonderland_early — lodging houses, and behind them a railway station.) 
However, she soon made out that she was in the ...
  7. wonderland_early — 16
`I wish I hadn't mentioned Dinah!' she said to herself in a melancholy tone. `Nobody seems to lik...
  8. wonderland_

In [22]:
ask("Who is the Caterpillar?")



Question: Who is the Caterpillar?
Answer:
The Caterpillar is a character that sits on top of a mushroom, quietly smoking a long hookah and engaging in a conversation with Alice.

Source chunks used (10):
  1. wonderland_court — WILLIAM,' to the Caterpillar, and the words all coming different, and then the Mock Turtle drew a 
l...
  2. wonderland_court — little sisters--they were learning to draw, you know--'  
 
`What did they draw?' said Alice, quite ...
  3. wonderland_court — and then, `we went to school in the sea. The master was an old Turtle--we used to call him Tortoise-...
  4. wonderland_court — 43
behind her, listening: so she went on, `--likely to win, that it's hardly worth while finishing t...
  5. wonderland_court — 57
 
 
`Consider your verdict,' the King said to the jury.  
 
`Not yet, not yet!' the Rabbit hastil...
  6. wonderland_early — She stretched herself up on tiptoe, and peeped over the edge of the mushroom, and her eyes 
immediat...
  7. wonderland_early — beg

In [23]:
ask("How does the trial of the Knave of Hearts reveal the absurd logic of Wonderland's justice system?")



Question: How does the trial of the Knave of Hearts reveal the absurd logic of Wonderland's justice system?
Answer:
I do not know.

Source chunks used (10):
  1. wonderland_court — this was of very little use, as it left no mark on the slate.  
 
`Herald, read the accusation!' sai...
  2. wonderland_court — `The trial cannot proceed,' said the King in a very grave voice, `until all the jurymen are back in ...
  3. wonderland_court — on which the wretched Hatter trembled so, that he shook both his shoes off.  
 
`Give your evidence,...
  4. wonderland_court — childhood: and how she would gather about her other little children, and make THEIR eyes bright and ...
  5. wonderland_court — players to be executed for having missed their turns, and she did not like the look of things at all...
  6. wonderland_early — waving the other paw, `lives a March Hare. Visit either you like: they're both mad.'...
  7. wonderland_early — many out-of-the-way things had happened lately, that Alice had beg

## 12 — Key Takeaways

1. **Liu et al. 2023** proved the U-shaped curve: LLMs attend to Position 1 & N, ignore the middle — even GPT-4.
2. **MergerRetriever** (`langchain_classic`) enables domain-separated multi-source retrieval.
3. **LongContextReorder** directly implements the paper's recommendation: push best docs to start & end.
4. **2026 stack**: `BAAI/bge-m3` embeddings, `gpt-4o-mini` LLM, LCEL chain, correct modular imports.
5. **Corpus**: *Alice's Adventures in Wonderland* (public domain, loaded from a real PDF via
   `PyPDFLoader`) replaces the earlier sample-text and 3_Idiots.pdf approaches — no copyright
   concerns, and it demonstrates real PDF ingestion end-to-end for this video.

### Optional: use your own PDF
```python
import os
os.environ['ALICE_PDF'] = r'/path/to/your_book.pdf'
```

> *Complete annotated notebook on GitHub — link in the description.*  
> *Today's recording uses clean code; GitHub has full documentation.*
